# ArcFace Face Recognition Training Pipeline
### Stage 3 of "ANGREZ PART" — Face Analysis & Identity Matching Pipeline

This notebook trains an **ArcFace** (Additive Angular Margin Loss) face recognition model
that plugs directly into Stage 3 of your pipeline:

```
Person Crop -> Face Detection -> [FACE RECOGNITION (this notebook)] -> Person ReID -> Database Matching (FAISS)
```

**What this notebook does, end to end:**
1. Installs dependencies
2. Downloads a face dataset **directly from Kaggle** (Kaggle API)
3. Detects & aligns faces (MTCNN) so the recognition model sees clean, centered faces — important for masked/low-res CCTV crops
4. Builds an **IResNet-50 backbone + ArcMarginProduct (ArcFace) head**
5. Trains with **checkpointing before/inside the training loop** (resume-safe, best-model tracking)
6. Evaluates verification accuracy (cosine similarity, ROC/AUC + best threshold)
7. Exports a clean **512-D embedding extractor** (TorchScript) ready to drop into your Stage 3 -> Stage 5 (FAISS) pipeline

> Runtime: Colab, GPU strongly recommended (Runtime -> Change runtime type -> GPU, ideally T4/A100).

---


## 1. Setup & Installation

In [ ]:
!pip install -q kaggle facenet-pytorch torch torchvision torchmetrics scikit-learn tqdm matplotlib opencv-python-headless
print("Dependencies installed.")


In [ ]:
import os
import io
import json
import time
import random
import shutil
import zipfile
import math
from pathlib import Path

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, random_split
import torchvision
from torchvision import transforms
from torchvision.datasets import ImageFolder
from sklearn.metrics import roc_curve, auc
from tqdm.auto import tqdm
import matplotlib.pyplot as plt

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", DEVICE)


## 2. Import Dataset from Kaggle

Upload your `kaggle.json` API token (Kaggle -> Account -> Create New API Token).

By default this pulls **`hereisburak/pins-face-recognition`** (105 identities, multiple
images each — a solid, lightweight dataset for training/validating a face-recognition
embedding model in Colab). Change `KAGGLE_DATASET` to any other Kaggle face dataset slug
(e.g. a CASIA-WebFace or VGGFace2 mirror) if you want a larger-scale run — just make sure
it is organized (or organizable) as `one folder per identity`.


In [ ]:
from google.colab import files

print("Please upload your kaggle.json (Kaggle -> Account -> Create New API Token)")
uploaded = files.upload()

os.makedirs("/root/.kaggle", exist_ok=True)
shutil.copy("kaggle.json", "/root/.kaggle/kaggle.json")
os.chmod("/root/.kaggle/kaggle.json", 0o600)
print("Kaggle credentials configured.")


In [ ]:
# ---- Configure dataset here ----
KAGGLE_DATASET = "hereisburak/pins-face-recognition"   # <-- change this slug for a different dataset
DATA_ROOT = Path("/content/data")
RAW_DIR = DATA_ROOT / "raw"
RAW_DIR.mkdir(parents=True, exist_ok=True)

!kaggle datasets download -d {KAGGLE_DATASET} -p {RAW_DIR} --unzip
print("Dataset downloaded and extracted to:", RAW_DIR)

# Peek at folder structure
for p in list(RAW_DIR.iterdir())[:10]:
    print(p)


In [ ]:
# Auto-detect the actual identity-folder root (Kaggle zips often nest one extra folder deep)
def find_identity_root(root: Path, min_subdirs: int = 5):
    candidates = [root] + [p for p in root.rglob('*') if p.is_dir()]
    best, best_count = None, 0
    for c in candidates:
        subdirs = [d for d in c.iterdir() if d.is_dir()] if c.is_dir() else []
        if len(subdirs) > best_count:
            best, best_count = c, len(subdirs)
    if best is None or best_count < min_subdirs:
        raise RuntimeError("Could not auto-detect an identity-folder structure. "
                            "Please point IDENTITY_ROOT manually at the folder containing one subfolder per person.")
    return best

IDENTITY_ROOT = find_identity_root(RAW_DIR)
NUM_IDENTITIES_RAW = len([d for d in IDENTITY_ROOT.iterdir() if d.is_dir()])
print("Detected identity root:", IDENTITY_ROOT)
print("Number of identity folders found:", NUM_IDENTITIES_RAW)


## 3. Face Detection & Alignment (MTCNN)

CCTV crops are noisy — off-angle, masked, low resolution. Feeding *aligned, tightly cropped*
faces into ArcFace training (rather than raw photos) is the single biggest lever for accuracy,
so we run every image through MTCNN first and rebuild a clean `aligned/<identity>/*.jpg` dataset.
Images where no face is detected are skipped (logged) rather than silently corrupting a class.


In [ ]:
from facenet_pytorch import MTCNN

mtcnn = MTCNN(image_size=112, margin=20, post_process=True, device=DEVICE, select_largest=True)

ALIGNED_DIR = DATA_ROOT / "aligned"
ALIGNED_DIR.mkdir(parents=True, exist_ok=True)

from PIL import Image

skipped = 0
kept = 0
identity_dirs = [d for d in IDENTITY_ROOT.iterdir() if d.is_dir()]

for identity_dir in tqdm(identity_dirs, desc="Aligning identities"):
    out_dir = ALIGNED_DIR / identity_dir.name
    out_dir.mkdir(parents=True, exist_ok=True)
    img_paths = [p for p in identity_dir.rglob('*') if p.suffix.lower() in ('.jpg', '.jpeg', '.png')]
    for img_path in img_paths:
        try:
            img = Image.open(img_path).convert("RGB")
            face_tensor = mtcnn(img)  # returns aligned 112x112 tensor in [-1, 1], or None
            if face_tensor is None:
                skipped += 1
                continue
            face_np = ((face_tensor.permute(1, 2, 0).numpy() * 0.5 + 0.5) * 255).astype("uint8")
            Image.fromarray(face_np).save(out_dir / (img_path.stem + ".jpg"), quality=95)
            kept += 1
        except Exception as e:
            skipped += 1

print(f"Aligned faces kept: {kept} | skipped (no face / corrupt): {skipped}")

# Drop identities that ended up with too few usable images (need >=2 for train/val split per class)
MIN_IMAGES_PER_IDENTITY = 2
for identity_dir in list(ALIGNED_DIR.iterdir()):
    if identity_dir.is_dir() and len(list(identity_dir.glob('*.jpg'))) < MIN_IMAGES_PER_IDENTITY:
        shutil.rmtree(identity_dir)

NUM_CLASSES = len([d for d in ALIGNED_DIR.iterdir() if d.is_dir()])
print("Final usable identity count:", NUM_CLASSES)


## 4. Datasets & Dataloaders

In [ ]:
IMG_SIZE = 112
BATCH_SIZE = 64
VAL_SPLIT = 0.1
NUM_WORKERS = 2

train_transform = transforms.Compose([
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2),
    transforms.RandomApply([transforms.GaussianBlur(kernel_size=3)], p=0.2),  # simulate CCTV blur
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.5, 0.5, 0.5], std=[0.5, 0.5, 0.5]),
])

val_transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.5, 0.5, 0.5], std=[0.5, 0.5, 0.5]),
])

full_dataset = ImageFolder(str(ALIGNED_DIR))
class_to_idx = full_dataset.class_to_idx
idx_to_class = {v: k for k, v in class_to_idx.items()}
NUM_CLASSES = len(class_to_idx)
print("Classes:", NUM_CLASSES, "| Total images:", len(full_dataset))

class TransformedSubset(Dataset):
    def __init__(self, subset, transform):
        self.subset = subset
        self.transform = transform
    def __len__(self):
        return len(self.subset)
    def __getitem__(self, idx):
        img, label = self.subset[idx]
        return self.transform(img), label

val_size = max(1, int(len(full_dataset) * VAL_SPLIT))
train_size = len(full_dataset) - val_size
train_subset, val_subset = random_split(full_dataset, [train_size, val_size],
                                         generator=torch.Generator().manual_seed(SEED))

train_ds = TransformedSubset(train_subset, train_transform)
val_ds = TransformedSubset(val_subset, val_transform)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,
                           num_workers=NUM_WORKERS, pin_memory=True, drop_last=True)
val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False,
                         num_workers=NUM_WORKERS, pin_memory=True)

print(f"Train samples: {len(train_ds)} | Val samples: {len(val_ds)}")


## 5. Model — IResNet Backbone + ArcFace (ArcMarginProduct) Head

- Backbone: ResNet-IR style (BatchNorm + PReLU, no bias in conv) producing a **512-D embedding**
  (same shape your pipeline diagram expects for Stage 3's `512-D Face Embedding`).
- Head: **ArcMarginProduct** — the actual "ArcFace" contribution (additive angular margin
  applied to the softmax logits), which is what makes the learned embeddings tightly
  clustered per identity and well-separated across identities.


In [ ]:
class IRBlock(nn.Module):
    def __init__(self, in_c, out_c, stride=1):
        super().__init__()
        self.bn0 = nn.BatchNorm2d(in_c)
        self.conv1 = nn.Conv2d(in_c, out_c, 3, 1, 1, bias=False)
        self.bn1 = nn.BatchNorm2d(out_c)
        self.prelu = nn.PReLU(out_c)
        self.conv2 = nn.Conv2d(out_c, out_c, 3, stride, 1, bias=False)
        self.bn2 = nn.BatchNorm2d(out_c)
        self.downsample = None
        if stride != 1 or in_c != out_c:
            self.downsample = nn.Sequential(
                nn.Conv2d(in_c, out_c, 1, stride, bias=False),
                nn.BatchNorm2d(out_c),
            )

    def forward(self, x):
        identity = x
        out = self.bn0(x)
        out = self.conv1(out)
        out = self.bn1(out)
        out = self.prelu(out)
        out = self.conv2(out)
        out = self.bn2(out)
        if self.downsample is not None:
            identity = self.downsample(x)
        return out + identity


class IResNet(nn.Module):
    """Compact IResNet-style backbone -> 512-D embedding."""
    def __init__(self, layers=(3, 4, 14, 3), embedding_size=512):
        super().__init__()
        self.input = nn.Sequential(
            nn.Conv2d(3, 64, 3, 1, 1, bias=False),
            nn.BatchNorm2d(64),
            nn.PReLU(64),
        )
        self.layer1 = self._make_layer(64, 64, layers[0], stride=2)
        self.layer2 = self._make_layer(64, 128, layers[1], stride=2)
        self.layer3 = self._make_layer(128, 256, layers[2], stride=2)
        self.layer4 = self._make_layer(256, 512, layers[3], stride=2)
        self.bn_out = nn.BatchNorm2d(512)
        self.dropout = nn.Dropout(0.4)
        # 112x112 input -> after 4 stride-2 stages -> 7x7 feature map
        self.fc = nn.Linear(512 * 7 * 7, embedding_size)
        self.features_bn = nn.BatchNorm1d(embedding_size)

    def _make_layer(self, in_c, out_c, blocks, stride):
        layers = [IRBlock(in_c, out_c, stride)]
        for _ in range(1, blocks):
            layers.append(IRBlock(out_c, out_c, 1))
        return nn.Sequential(*layers)

    def forward(self, x):
        x = self.input(x)
        x = self.layer1(x)
        x = self.layer2(x)
        x = self.layer3(x)
        x = self.layer4(x)
        x = self.bn_out(x)
        x = self.dropout(x)
        x = torch.flatten(x, 1)
        x = self.fc(x)
        x = self.features_bn(x)
        return x


In [ ]:
class ArcMarginProduct(nn.Module):
    """ArcFace additive angular margin head.
    Ref: Deng et al., "ArcFace: Additive Angular Margin Loss for Deep Face Recognition" (2019)
    """
    def __init__(self, in_features, out_features, s=64.0, m=0.5, easy_margin=False):
        super().__init__()
        self.in_features = in_features
        self.out_features = out_features
        self.s = s
        self.m = m
        self.weight = nn.Parameter(torch.FloatTensor(out_features, in_features))
        nn.init.xavier_uniform_(self.weight)

        self.easy_margin = easy_margin
        self.cos_m = math.cos(m)
        self.sin_m = math.sin(m)
        self.th = math.cos(math.pi - m)
        self.mm = math.sin(math.pi - m) * m

    def forward(self, embeddings, labels):
        cosine = F.linear(F.normalize(embeddings), F.normalize(self.weight))
        sine = torch.sqrt((1.0 - torch.clamp(cosine ** 2, 0, 1)))
        phi = cosine * self.cos_m - sine * self.sin_m
        if self.easy_margin:
            phi = torch.where(cosine > 0, phi, cosine)
        else:
            phi = torch.where(cosine > self.th, phi, cosine - self.mm)

        one_hot = torch.zeros_like(cosine)
        one_hot.scatter_(1, labels.view(-1, 1).long(), 1)
        output = one_hot * phi + (1.0 - one_hot) * cosine
        output = output * self.s
        return output


In [ ]:
EMBEDDING_SIZE = 512
ARC_S = 64.0
ARC_M = 0.5

backbone = IResNet(layers=(3, 4, 14, 3), embedding_size=EMBEDDING_SIZE).to(DEVICE)
arc_head = ArcMarginProduct(EMBEDDING_SIZE, NUM_CLASSES, s=ARC_S, m=ARC_M).to(DEVICE)

n_params = sum(p.numel() for p in backbone.parameters()) / 1e6
print(f"Backbone parameters: {n_params:.2f}M | Classes: {NUM_CLASSES}")


## 6. Training Configuration

In [ ]:
EPOCHS = 40
BASE_LR = 0.1
WEIGHT_DECAY = 5e-4
MOMENTUM = 0.9
WARMUP_EPOCHS = 3
USE_AMP = True
LABEL_SMOOTHING = 0.0  # ArcFace margin already regularizes; keep 0 unless you see overfitting

criterion = nn.CrossEntropyLoss(label_smoothing=LABEL_SMOOTHING)

optimizer = torch.optim.SGD(
    list(backbone.parameters()) + list(arc_head.parameters()),
    lr=BASE_LR, momentum=MOMENTUM, weight_decay=WEIGHT_DECAY, nesterov=True,
)

def lr_lambda(epoch):
    if epoch < WARMUP_EPOCHS:
        return (epoch + 1) / WARMUP_EPOCHS
    progress = (epoch - WARMUP_EPOCHS) / max(1, (EPOCHS - WARMUP_EPOCHS))
    return 0.5 * (1 + math.cos(math.pi * progress))

scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda)
scaler = torch.cuda.amp.GradScaler(enabled=USE_AMP)

print("Training config ready:", dict(EPOCHS=EPOCHS, BASE_LR=BASE_LR, WEIGHT_DECAY=WEIGHT_DECAY))


## 7. Checkpointing Utilities (placed *before* the training loop)

- `save_checkpoint`: saves model/head/optimizer/scheduler/scaler state + epoch + best-metric,
  so training can be **resumed exactly** after a Colab disconnect.
- `load_checkpoint`: restores everything if a checkpoint already exists.
- A **best-model** checkpoint is tracked separately by validation accuracy.


In [ ]:
CKPT_DIR = Path("/content/checkpoints")
CKPT_DIR.mkdir(parents=True, exist_ok=True)
LAST_CKPT_PATH = CKPT_DIR / "last_checkpoint.pth"
BEST_CKPT_PATH = CKPT_DIR / "best_checkpoint.pth"

# Optional: mount Google Drive so checkpoints survive a full runtime reset.
try:
    from google.colab import drive
    drive.mount('/content/drive', force_remount=False)
    DRIVE_CKPT_DIR = Path("/content/drive/MyDrive/arcface_checkpoints")
    DRIVE_CKPT_DIR.mkdir(parents=True, exist_ok=True)
    print("Google Drive mounted — checkpoints will also be mirrored to:", DRIVE_CKPT_DIR)
except Exception as e:
    DRIVE_CKPT_DIR = None
    print("Drive not mounted (continuing with local /content checkpoints only):", e)


def save_checkpoint(state: dict, is_best: bool, path=LAST_CKPT_PATH, best_path=BEST_CKPT_PATH):
    torch.save(state, path)
    if DRIVE_CKPT_DIR is not None:
        torch.save(state, DRIVE_CKPT_DIR / path.name)
    if is_best:
        torch.save(state, best_path)
        if DRIVE_CKPT_DIR is not None:
            torch.save(state, DRIVE_CKPT_DIR / best_path.name)


def load_checkpoint(path=LAST_CKPT_PATH):
    if not Path(path).exists():
        return None
    ckpt = torch.load(path, map_location=DEVICE)
    backbone.load_state_dict(ckpt["backbone_state"])
    arc_head.load_state_dict(ckpt["head_state"])
    optimizer.load_state_dict(ckpt["optimizer_state"])
    scheduler.load_state_dict(ckpt["scheduler_state"])
    scaler.load_state_dict(ckpt["scaler_state"])
    print(f"Resumed from checkpoint at epoch {ckpt['epoch']} (best_val_acc so far: {ckpt['best_val_acc']:.4f})")
    return ckpt

print("Checkpoint utilities ready. Checkpoints will be written to:", CKPT_DIR)


## 8. Training Loop (with checkpointing every epoch + resume support)

In [ ]:
@torch.no_grad()
def evaluate_classification(loader):
    backbone.eval()
    arc_head.eval()
    correct, total = 0, 0
    for imgs, labels in loader:
        imgs, labels = imgs.to(DEVICE), labels.to(DEVICE)
        embeddings = backbone(imgs)
        logits = F.linear(F.normalize(embeddings), F.normalize(arc_head.weight))
        preds = logits.argmax(dim=1)
        correct += (preds == labels).sum().item()
        total += labels.size(0)
    return correct / max(1, total)


def train_one_epoch(epoch):
    backbone.train()
    arc_head.train()
    running_loss, correct, total = 0.0, 0, 0
    pbar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{EPOCHS}")
    for imgs, labels in pbar:
        imgs, labels = imgs.to(DEVICE), labels.to(DEVICE)
        optimizer.zero_grad(set_to_none=True)

        with torch.cuda.amp.autocast(enabled=USE_AMP):
            embeddings = backbone(imgs)
            logits = arc_head(embeddings, labels)
            loss = criterion(logits, labels)

        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()

        running_loss += loss.item() * imgs.size(0)
        preds = logits.argmax(dim=1)
        correct += (preds == labels).sum().item()
        total += labels.size(0)
        pbar.set_postfix(loss=running_loss/total, acc=correct/total)

    return running_loss / total, correct / total


history = {"train_loss": [], "train_acc": [], "val_acc": [], "lr": []}
start_epoch = 0
best_val_acc = 0.0

# ---- Resume automatically if a checkpoint already exists ----
ckpt = load_checkpoint(LAST_CKPT_PATH)
if ckpt is not None:
    start_epoch = ckpt["epoch"] + 1
    best_val_acc = ckpt["best_val_acc"]
    history = ckpt.get("history", history)

for epoch in range(start_epoch, EPOCHS):
    t0 = time.time()
    train_loss, train_acc = train_one_epoch(epoch)
    val_acc = evaluate_classification(val_loader)
    scheduler.step()

    current_lr = optimizer.param_groups[0]["lr"]
    history["train_loss"].append(train_loss)
    history["train_acc"].append(train_acc)
    history["val_acc"].append(val_acc)
    history["lr"].append(current_lr)

    is_best = val_acc > best_val_acc
    best_val_acc = max(val_acc, best_val_acc)

    # ---- Checkpoint every epoch (this is the checkpointer the pipeline needs) ----
    save_checkpoint({
        "epoch": epoch,
        "backbone_state": backbone.state_dict(),
        "head_state": arc_head.state_dict(),
        "optimizer_state": optimizer.state_dict(),
        "scheduler_state": scheduler.state_dict(),
        "scaler_state": scaler.state_dict(),
        "best_val_acc": best_val_acc,
        "history": history,
        "class_to_idx": class_to_idx,
        "num_classes": NUM_CLASSES,
        "embedding_size": EMBEDDING_SIZE,
    }, is_best=is_best)

    dt = time.time() - t0
    print(f"[Epoch {epoch+1}/{EPOCHS}] loss={train_loss:.4f} train_acc={train_acc:.4f} "
          f"val_acc={val_acc:.4f} best_val_acc={best_val_acc:.4f} lr={current_lr:.5f} ({dt:.1f}s)"
          + ("  <-- new best, saved" if is_best else ""))

print("Training complete. Best validation accuracy:", best_val_acc)


## 9. Verification Evaluation (Cosine Similarity, ROC/AUC, Best Threshold)

Classification accuracy tells you how well the model separates the *training identities*.
What actually matters for your CCTV pipeline is **verification** performance: given two
face embeddings, does cosine similarity correctly say "same person" / "different person"?
This cell builds genuine/impostor pairs from the validation set and reports ROC-AUC and the
best operating threshold — the number you should plug into Stage 5 (FAISS similarity search)
as your match/no-match cutoff.


In [ ]:
@torch.no_grad()
def extract_embeddings(loader):
    backbone.eval()
    all_embs, all_labels = [], []
    for imgs, labels in loader:
        imgs = imgs.to(DEVICE)
        embs = backbone(imgs)
        embs = F.normalize(embs, dim=1)
        all_embs.append(embs.cpu())
        all_labels.append(labels)
    return torch.cat(all_embs), torch.cat(all_labels)

val_embs, val_labels = extract_embeddings(val_loader)

# Build pairs: all genuine pairs available + an equal number of random impostor pairs
genuine_scores, impostor_scores = [], []
labels_np = val_labels.numpy()
embs_np = val_embs.numpy()

for cls in np.unique(labels_np):
    idxs = np.where(labels_np == cls)[0]
    for i in range(len(idxs)):
        for j in range(i + 1, len(idxs)):
            sim = float(np.dot(embs_np[idxs[i]], embs_np[idxs[j]]))
            genuine_scores.append(sim)

rng = np.random.default_rng(SEED)
n_impostor = max(len(genuine_scores), 500)
for _ in range(n_impostor):
    i, j = rng.choice(len(labels_np), 2, replace=False)
    if labels_np[i] != labels_np[j]:
        impostor_scores.append(float(np.dot(embs_np[i], embs_np[j])))

y_true = np.array([1] * len(genuine_scores) + [0] * len(impostor_scores))
y_score = np.array(genuine_scores + impostor_scores)

fpr, tpr, thresholds = roc_curve(y_true, y_score)
roc_auc = auc(fpr, tpr)
best_idx = np.argmax(tpr - fpr)
best_threshold = thresholds[best_idx]

print(f"Verification ROC-AUC: {roc_auc:.4f}")
print(f"Recommended cosine-similarity match threshold for FAISS stage: {best_threshold:.4f}")

plt.figure(figsize=(5, 5))
plt.plot(fpr, tpr, label=f"ROC (AUC={roc_auc:.3f})")
plt.plot([0, 1], [0, 1], linestyle="--", color="gray")
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("Face Verification ROC Curve")
plt.legend()
plt.show()


## 10. Training Curves

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4))
axes[0].plot(history["train_loss"]); axes[0].set_title("Train Loss"); axes[0].set_xlabel("Epoch")
axes[1].plot(history["train_acc"], label="train"); axes[1].plot(history["val_acc"], label="val")
axes[1].set_title("Accuracy"); axes[1].set_xlabel("Epoch"); axes[1].legend()
axes[2].plot(history["lr"]); axes[2].set_title("Learning Rate"); axes[2].set_xlabel("Epoch")
plt.tight_layout()
plt.show()


## 11. Export for Pipeline Integration (Stage 3: Face Recognition)

This loads the **best** checkpoint and exports:
- `arcface_backbone.pt` — TorchScript module: `aligned_face_tensor -> 512-D embedding`, ready
  to call for every `Detected Face` your Stage 2 (Face Detection) produces.
- `class_to_idx.json` — identity label mapping (useful if you also want closed-set classification,
  though for open-set CCTV matching you'll mainly use the embeddings + FAISS from Stage 5).


In [ ]:
best_ckpt = torch.load(BEST_CKPT_PATH, map_location=DEVICE)
backbone.load_state_dict(best_ckpt["backbone_state"])
backbone.eval()

EXPORT_DIR = Path("/content/export")
EXPORT_DIR.mkdir(exist_ok=True)

example_input = torch.randn(1, 3, IMG_SIZE, IMG_SIZE).to(DEVICE)
traced_backbone = torch.jit.trace(backbone, example_input)
traced_backbone.save(str(EXPORT_DIR / "arcface_backbone.ts"))

with open(EXPORT_DIR / "class_to_idx.json", "w") as f:
    json.dump(class_to_idx, f, indent=2)

with open(EXPORT_DIR / "config.json", "w") as f:
    json.dump({
        "embedding_size": EMBEDDING_SIZE,
        "img_size": IMG_SIZE,
        "arc_s": ARC_S,
        "arc_m": ARC_M,
        "best_val_acc": best_ckpt["best_val_acc"],
        "recommended_match_threshold": float(best_threshold),
    }, f, indent=2)

print("Exported to:", EXPORT_DIR)
for p in EXPORT_DIR.iterdir():
    print(" -", p)


In [ ]:
def get_face_embedding(aligned_face_pil, model=traced_backbone, device=DEVICE):
    """Stage 3 entry point: aligned 112x112 PIL face -> normalized 512-D embedding (numpy array).
    Feed this the output of your Stage 2 (Face Detection / alignment) crops.
    """
    tfm = transforms.Compose([
        transforms.Resize((IMG_SIZE, IMG_SIZE)),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.5, 0.5, 0.5], std=[0.5, 0.5, 0.5]),
    ])
    x = tfm(aligned_face_pil).unsqueeze(0).to(device)
    with torch.no_grad():
        emb = model(x)
        emb = F.normalize(emb, dim=1)
    return emb.cpu().numpy().squeeze(0)

# Quick sanity check on one validation image
sample_img_path = next(ALIGNED_DIR.iterdir())
sample_img_path = next(sample_img_path.glob("*.jpg"))
sample_face = Image.open(sample_img_path).convert("RGB")
emb = get_face_embedding(sample_face)
print("Embedding shape:", emb.shape, "| L2 norm:", np.linalg.norm(emb))


## 12. Download Artifacts

Zips the exported model + configs so you can drop them straight into your Stage 3 -> Stage 5
FAISS matching service.


In [ ]:
zip_path = "/content/arcface_export.zip"
shutil.make_archive("/content/arcface_export", "zip", EXPORT_DIR)

from google.colab import files as colab_files
colab_files.download(zip_path)
print("Download triggered:", zip_path)


---
### Notes on getting the best accuracy
- **More data per identity beats a bigger model** — if `val_acc` plateaus low, the fastest fix is usually more images per identity, not more epochs.
- If you swap in a larger dataset (CASIA-WebFace / VGGFace2 scale), increase `layers` in `IResNet` (e.g. `(3, 13, 30, 3)` for a deeper IResNet-100) and train for more epochs with a larger batch size.
- If training loss stalls at the very start, lower `ARC_M` (e.g. `0.3`) for the first few epochs, then raise it back to `0.5` — a known ArcFace trick for unstable early convergence.
- Keep `USE_AMP = True` on Colab GPUs — it roughly halves training time with negligible accuracy impact.
- The checkpointing above means you can stop the notebook at any point and simply re-run the training-loop cell — it will auto-resume from `last_checkpoint.pth`.
